In [1]:
import tkinter
import tkintermapview
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely
import statistics as stats
import fiona
from shapely.geometry import mapping, Polygon
from shapely.ops import transform

region = 'west-bengal'
path_shp = 'C:/IMAGE/Foliage/test/estimation/' + region + '.shp'

# create tkinter window
root_tk = tkinter.Tk()
root_tk.geometry(f"{1000}x{700}")
root_tk.title("map_view_polygon_example.py")

# create map widget
map_widget = tkintermapview.TkinterMapView(root_tk, width=1000, height=700, corner_radius=0)
map_widget.pack(fill="both", expand=True)

# set current widget position and zoom
map_widget.set_position(12.979831276192142, 77.59109621564363)  # Vidhan Sabha, Bangalore, India
map_widget.set_zoom(17)

points = []
polygon_1 = None
polygon_2 = None
coord_polygon = []

df_forGeo = pd.DataFrame()

def polygon_click(polygon):
    #print(f"polygon clicked - text: {polygon.name}")
    #print(str(polygon))
    pass

#def left_click_event(coordinates_tuple):
#    print("Left click event with coordinates:", coordinates_tuple)
#    return coordinates_tuple

def add_marker_event(coords):
    global points
    #print("Add marker:", coords)
    new_marker = map_widget.set_marker(coords[0], coords[1], text="")
    points.append(coords)
    #print(points)

def create_polygon_event():
    map_widget.delete_all_polygon()
    
    global points
    #print(points)
    
    c_lat = [x[0] for x in points]
    c_lon = [x[1] for x in points]
    c_P = (min(c_lat),min(c_lon))
    c_Q = (max(c_lat),min(c_lon))
    c_R = (max(c_lat),max(c_lon))
    c_S = (min(c_lat),max(c_lon))

    poly_corners_1 = [c_P,c_Q,c_R,c_S]

    polygon_1 = map_widget.set_polygon(poly_corners_1, 
                                    #list_coords,
                                    # fill_color=None,
                                    # outline_color="red",
                                    # border_width=12,
                                    command=polygon_click,
                                    name="switzerland_polygon")
    polygon_1.add_position(0, 0, index=5)
    polygon_1.remove_position(0, 0)

    global coord_polygon
    coord_polygon = poly_corners_1.copy()


def create_jagged_polygon_event():
    map_widget.delete_all_polygon()

    global points
    #print(points)

    c_lat = [p[0] for p in points]
    c_lon = [p[1] for p in points]
    c_lat.sort()
    c_lon.sort()
    L = list()
    R = list()
    A = list()
    B = list()
    
    #print(stats.median(c_lat), stats.median_low(c_lat), stats.median_high(c_lat))
    points_medianLat = [p for p in points] # REMOVED CONDITION: " if p[0]==stats.median(c_lat)"
    points_medianLat.sort()
    c_medianLat = [p[0] for p in points_medianLat]
    c_medianLon = [p[1] for p in points_medianLat]
    c_medianLat.sort()
    c_medianLon.sort()
    for p in points_medianLat:
        # Find the leftmost point L
        if p[1] == min(c_medianLon):
            L = p
        
        # Find the rightmost point R
        if p[1] == max(c_medianLon):
            R = p
    
    # Partition the points into A, the set of points below LR; and B, the set of points above LR
    # LR equation: X = Y * (R[0] - L[0]) / (R[1] - L[1])
    for p in points:
        #print(p, points_medianLat)
        cross_product = np.cross(np.array(list(p))-np.array(list(L)), np.array(list(R))-np.array(list(L)))
        if cross_product >= 0:
            A.append(p)
        if cross_product < 0:
            B.append(p)
    
    
    # Sort A by x-coordinate (increasing)
    A.sort(key=lambda p: p[1], reverse=False)

    #Sort B by x-coordinate (decreasing)
    B.sort(key=lambda p: p[1], reverse=True)

    #Return the polygon defined by L, the points in A in order, R, the points of B in order
    poly_corners_2 = []
    poly_corners_2.append(L)
    poly_corners_2.extend(A)
    poly_corners_2.append(R)
    poly_corners_2.extend(B)
    poly_corners_2.append(poly_corners_2[0])

    #print(poly_corners_2)

    polygon_2 = map_widget.set_polygon(poly_corners_2, 
                                    #list_coords,
                                    # fill_color=None,
                                    # outline_color="red",
                                    # border_width=12,
                                    command=polygon_click,
                                    name="switzerland_polygon")
    polygon_2.add_position(0, 0, index=5)
    polygon_2.remove_position(0, 0)

    global coord_polygon
    coord_polygon = poly_corners_2.copy()


def delete_all_polygons_event():
    global points
    points = []
    map_widget.delete_all_polygon()
    map_widget.delete_all_marker()

    global coord_polygon
    coord_polygon = []


def flip(x, y):
    """Flips the x and y coordinate values"""
    return y, x


def save_as_shapefile_event():
    global coord_polygon
    poly = Polygon(tuple(coord_polygon))

    poly = transform(flip, poly)

    # Define a polygon feature geometry with one attribute
    schema = {
        'geometry': 'Polygon',
        'properties': {'id': 'int'},
    }

    # Write a new Shapefile
    with fiona.open(path_shp, 'w', 'ESRI Shapefile', schema) as c:
        ## If there are multiple geometries, put the "for" loop here
        c.write({
            'geometry': mapping(poly),
            'properties': {'id': 123},
        })

   
#map_widget.add_left_click_map_command(left_click_event)

#for i in range(4):
#    a = left_click_event()
#    points.append(a)

map_widget.add_right_click_menu_command(label="Add Marker",
                                        command=add_marker_event,
                                        pass_coords=True)

map_widget.add_right_click_menu_command(label="Create Polygon",
                                        command=create_polygon_event,
                                        pass_coords=False)

map_widget.add_right_click_menu_command(label="Create Jagged Polygon",
                                        command=create_jagged_polygon_event,
                                        pass_coords=False)

map_widget.add_right_click_menu_command(label="Delete All Polygons",
                                        command=delete_all_polygons_event,
                                        pass_coords=False)

map_widget.add_right_click_menu_command(label="Save As Shapefile (.shp)",
                                        command=save_as_shapefile_event,
                                        pass_coords=False)

#map_widget.set_tile_server("https://a.tile.openstreetmap.org/{z}/{x}/{y}.png")  # OpenStreetMap (default)
map_widget.set_tile_server("https://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}&s=Ga", max_zoom=22)  # google satellite

root_tk.mainloop()

In [ ]:
"""
#INITIAL TKINTER SCRIPT FOR LEARNING PURPOSES

import tkinter
import tkintermapview

# define position function
def set_position(map_widget):
    # set current widget position and zoom
    #map_widget.set_position(12.979831276192142, 77.59109621564363)  # Paris, France
    #map_widget.set_zoom(15)

    # set current widget position by address
    map_widget.set_address("colosseo, rome, italy")

    # set current widget position by address
    #marker_1 = map_widget.set_address("colosseo, rome, italy", marker=True)

    #print(marker_1)  # get position and text

    #marker_1.set_text("Colosseo in Rome")  # set new text
    # marker_1.set_position(48.860381, 2.338594)  # change position
    # marker_1.delete()

# create tkinter window
root_tk = tkinter.Tk()
root_tk.geometry(f"{1080}x{1080}")
root_tk.title("map_view_example.py")

# create map widget
map_widget = tkintermapview.TkinterMapView(root_tk, width=800, height=600, corner_radius=0)
map_widget.place(relx=0.5, rely=0.5, anchor=tkinter.CENTER)

# set current widget position and zoom
map_widget.set_position(12.979831276192142, 77.59109621564363)  # Vidhan Sabha, Bangalore, India
map_widget.set_zoom(17)

def add_marker_event(coords):
    print("Add marker:", coords)
    new_marker = map_widget.set_marker(coords[0], coords[1], text="new marker")

def left_click_event(coordinates_tuple):
    print("Left click event with coordinates:", coordinates_tuple)
    
map_widget.add_left_click_map_command(left_click_event)
    

map_widget.add_right_click_menu_command(label="Add Marker",
                                        command=add_marker_event,
                                        pass_coords=True)

# example tile sever:
#map_widget.set_tile_server("https://a.tile.openstreetmap.org/{z}/{x}/{y}.png")  # OpenStreetMap (default)
map_widget.set_tile_server("https://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}&s=Ga", max_zoom=22)  # google satellite

# example overlay tile server
#map_widget.set_overlay_tile_server("http://tiles.openseamap.org/seamark//{z}/{x}/{y}.png")  # sea-map overlay
#map_widget.set_overlay_tile_server("http://a.tiles.openrailwaymap.org/standard/{z}/{x}/{y}.png")  # railway infrastructure

root_tk.mainloop()
"""

In [ ]:
"""
import tkinter
import tkintermapview
import numpy as np

# create tkinter window
root_tk = tkinter.Tk()
root_tk.geometry(f"{1000}x{700}")
root_tk.title("map_view_polygon_example.py")

# create map widget
map_widget = tkintermapview.TkinterMapView(root_tk, width=1000, height=700, corner_radius=0)
map_widget.pack(fill="both", expand=True)

# set current widget position and zoom
#switzerland_marker = map_widget.set_address("Switzerland", marker=True, text="Switzerland")
#map_widget.set_zoom(8)
map_widget.set_position(12.979831276192142, 77.59109621564363)  # Vidhan Sabha, Bangalore, India
map_widget.set_zoom(17)


def polygon_click(polygon):
    print(f"polygon clicked - text: {polygon.name}")
    print(str(polygon))


list_coords = [(46.0732306, 6.0095215),
                                    (46.3393433, 6.2072754),
                                    (46.5890691, 6.1083984),
                                    (46.7624431, 6.4270020),
                                    (47.2717751, 7.0312500),
                                    (47.4726629, 6.9982910),
                                    (47.4057853, 7.3718262),
                                    (47.5468716, 7.9650879),
                                    (47.5691138, 8.4045410),
                                    (47.7540980, 8.6242676),
                                    (47.5691138, 9.4482422),
                                    (47.1897125, 9.5581055),
                                    (46.9352609, 9.8327637),
                                    (46.9727564, 10.4150391),
                                    (46.6418940, 10.4479980),
                                    (46.4605655, 10.0744629),
                                    (46.2786312, 10.1513672),
                                    (46.3469276, 9.5581055),
                                    (46.4454275, 9.3493652),
                                    (45.8211434, 8.9538574),
                                    (46.1037088, 8.6352539),
                                    (46.3696741, 8.3496094),
                                    (45.9740604, 7.9321289),
                                    (45.8900082, 7.0971680),
                                    (46.1417827, 6.8664551),
                                    (46.4151388, 6.7236328),
                                    (46.3772542, 6.4160156)]
'''
c_lat = [x[0] for x in list_coords]
c_lon = [x[1] for x in list_coords]
c_P = (min(c_lat),min(c_lon))
c_Q = (max(c_lat),min(c_lon))
c_R = (max(c_lat),max(c_lon))
c_S = (min(c_lat),max(c_lon))
'''

points = []
polygon_1 = None
polygon_2 = None

#def left_click_event(coordinates_tuple):
#    print("Left click event with coordinates:", coordinates_tuple)
#    return coordinates_tuple

def add_marker_event(coords):
    global points
    print("Add marker:", coords)
    new_marker = map_widget.set_marker(coords[0], coords[1], text="")
    points.append(coords)
    print(points)

def create_polygon_event():
    if polygon_2 is not None:
        polygon_2.delete()
    
    global points
    print(points)
    
    c_lat = [x[0] for x in points]
    c_lon = [x[1] for x in points]
    c_P = (min(c_lat),min(c_lon))
    c_Q = (max(c_lat),min(c_lon))
    c_R = (max(c_lat),max(c_lon))
    c_S = (min(c_lat),max(c_lon))

    polygon_1 = map_widget.set_polygon([c_P,c_Q,c_R,c_S], 
                                    #list_coords,
                                    # fill_color=None,
                                    # outline_color="red",
                                    # border_width=12,
                                    command=polygon_click,
                                    name="switzerland_polygon")
    polygon_1.add_position(0, 0, index=5)
    polygon_1.remove_position(0, 0)
    
def create_jagged_polygon_event():
    if polygon_1 is not None:
        polygon_1.delete()

    global points
    print(points)

    c_lat = [p[0] for p in points]
    c_lon = [p[1] for p in points]
    L = list()
    R = list()
    A = list()
    B = list()
    
    for p in points:
        # Find the leftmost point L
        if p[1] == min(c_lon):
            L = p
        
        # Find the rightmost point R
        if p[1] == max(c_lon):
            R = p

    # Partition the points into A, the set of points below LR, and B, the set of points above LR
    # LR equation: X = Y * (R[0] - L[0]) / (R[1] - L[1])

    v1 = (R[0] - L[0], R[1] - L[1])
    for p in points:
        v2 = (p[0] - L[0], p[1] - L[1])
        #cross_product = v1[0]*v2[1] - v1[1]*v2[0]
        cross_product = np.cross(np.array(list(p))-np.array(list(L)), np.array(list(R))-np.array(list(L)))
        #if (p[0] <= p[1] * (R[0] - L[0]) / (R[1] - L[1])):
            #or (p[0] <= L[0] and p[0] <= R[0]):
            #A.append(p)
        #if (p[0] >= p[1] * (R[0] - L[0]) / (R[1] - L[1])):
            #or (p[0] >= L[0] and p[0] >= R[0]):
            #B.append(p)
        if cross_product >= 0:
            A.append(p)
        if cross_product < 0:
            B.append(p)
    
    
    # Sort A by x-coordinate (increasing)
    A.sort(key=lambda p: p[1], reverse=False)

    #Sort B by x-coordinate (decreasing)
    B.sort(key=lambda p: p[1], reverse=True)

    #Return the polygon defined by L, the points in A in order, R, the points of B in order
    poly_corners = []
    poly_corners.append(L)
    poly_corners.extend(A)
    poly_corners.append(R)
    poly_corners.extend(B)
    poly_corners.append(poly_corners[0])

    print(poly_corners)

    polygon_2 = map_widget.set_polygon(poly_corners, 
                                    #list_coords,
                                    # fill_color=None,
                                    # outline_color="red",
                                    # border_width=12,
                                    command=polygon_click,
                                    name="switzerland_polygon")
    polygon_2.add_position(0, 0, index=5)
    polygon_2.remove_position(0, 0)

   
#map_widget.add_left_click_map_command(left_click_event)

#for i in range(4):
#    a = left_click_event()
#    points.append(a)

map_widget.add_right_click_menu_command(label="Add Marker",
                                        command=add_marker_event,
                                        pass_coords=True)

map_widget.add_right_click_menu_command(label="Create Polygon",
                                        command=create_polygon_event,
                                        pass_coords=False)

map_widget.add_right_click_menu_command(label="Create Jagged Polygon",
                                        command=create_jagged_polygon_event,
                                        pass_coords=False)


# polygon_1.remove_position(46.3772542, 6.4160156)
# polygon_1.add_position(0, 0, index=5)
# polygon_1.delete()

#map_widget.set_tile_server("https://a.tile.openstreetmap.org/{z}/{x}/{y}.png")  # OpenStreetMap (default)
map_widget.set_tile_server("https://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}&s=Ga", max_zoom=22)  # google satellite

root_tk.mainloop()
"""